In [ ]:
%%capture
!pip install -U kaleido
!pip install pandas_profiling

In [ ]:
import numpy as np
import pandas as pd 
from pprint import pprint
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
pio.templates.default = 'simple_white'
mycolors = ['#265C4B','#146551','#007566','#589A8D','#8FC1B5']

from pandas_profiling import ProfileReport
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,models,Sequential

<a id="4"></a>
# **<center><span style="color:#00BFC4;"> Introduction </span></center>**

Starting with pandas, this notebook describes the commonly used functions and methods used in data analysis, and performed a simple visualization of the data using plotly. In addition, I performed an overall processing and prediction process for the space data set. A simple model was used and the accuracy rate was `78%`. If you find it helpful, please `upvote`  and  leave comments for me ! thank you!

<a id="4"></a>
# **<center><span style="color:#00BFC4;"> Setup </span></center>**

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

<a id="4"></a>
# **<center><span style="color:#00BFC4;"> Introduction to Pandas </span></center>**

It contains data structures and data manipulation tools designed to make `data cleaning and analysis` fast and convenient in Python. pandas is often used in tandem with numerical computing tools like NumPy and SciPy, analytical libraries like statsmodels and scikit-learn, and data visualization libraries like matplotlib.

There are two common data structures used in pandas：
* `Series`: a one-dimensional array-like object containing values and index
* `DataFrame`: represents a rectangular table of data and contains an ordered, named collection of columns, each of which can be a different value type (numeric, string, Boolean, etc.).

## <font color='#9966FF'> Series <font><a class='anchor' id='top'></a>

There are three ways to create a Series :
1. Only `values` are given and `index` are automatically created
2. `values` and `index` are given
3. from a `dict`

We get `values` from  **.values** and `index` from **.index**

In [ ]:
# 1 
my_series = pd.Series(['India','US','UK','Russia'])
# 2
my_series = pd.Series(['India','US','UK','Russia'],index=[1,2,3,4])
# 3
my_dict = {1:'India',2:'US',3:'UK',4:'Russia'}
my_series = pd.Series(my_dict)

print(my_series.values,"\n",my_series.index)

## <font color='#9966FF'> DataFrame <font><a class='anchor' id='top'></a>
There are two ways to create a Series :   
1. from a numpy array
2. from a dict

In [ ]:
# 1
data = np.random.randint(1,10,size=(3,4))
my_df = pd.DataFrame(data)
# 2
data = {'state': ['Ohio', 'Ohio', 'Ohio', 'Nevada', 'Nevada', 'Nevada'],
        'year': [2000, 2001, 2002, 2001, 2002, 2003],
        'pop': [1.5, 1.7, 3.6, 2.4, 2.9, 3.2]}
my_df = pd.DataFrame(data)

## <font color='#9966FF'> What do we do with pandas？ <font><a class='anchor' id='top'></a>
pandas features a number of functions for reading `tabular` data as a DataFrame object,`pandas.read_csv` is one of the most frequently used in Kaggle!

In [ ]:
# data loading
df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
# quick view on our data
df.head() # First five lines of data
df.sample() # sample a lines of data at random
df.tail() # last five lines of data
df.describe() # mathematical overview of the data

## <font color='#9966FF'> Data Preparation <font><a class='anchor' id='top'></a>
During the course of doing data analysis and modeling, a significant amount of time is spent on data preparation: loading, cleaning, transforming, and rearranging. Such tasks are often reported to take up 80% or more of an analyst's time. 

<a id="4.1"></a>
### <span style="color:#e76f51;"> Dealing With Missing Data</span>
As there may be various interference factors in the acquisition process of data to be analyzed, it is `common` for data to have missing values during data analysis


In [ ]:
# Check all rows for missing values, it will return a Series
df.isna().sum()
# print(type(df.isna().sum()))

After the missing value is found, it can carry out the missing value elimination, the missing value filling and other pretreatment methods

In [ ]:
# missing value elimination

# df.dropna(axis='columns') #drop columns with missing values 
df.dropna(axis='index')#drop rows with missing values 

# missing value filling
# for numerical features eg;'RoomService'
from sklearn.impute import SimpleImputer,KNNImputer
imputer = SimpleImputer(strategy='mean')
df['RoomService'] = imputer.fit_transform(df['RoomService'].to_numpy().reshape(-1,1))
df['RoomService'] = df['RoomService'].apply(lambda x : round(x))
# for categorical features : eg:'Cabin'
df['Cabin'] = df['Cabin'].fillna(method="ffill") # or bfill

<a id="4.1"></a>
### <span style="color:#e76f51;"> Dealing With Duplicates</span>

In [ ]:
# check duplicates
df.duplicated().sum()
# drop duplicates
df.drop_duplicates() #Discard duplicate values for all columns
new_df = df.drop_duplicates(subset=['Age']) # Discard duplicate values for ‘Age' columns  ！just a example ,don't do this !

<a id="4.1"></a>
### <span style="color:#e76f51;"> Dealing With Outliers</span>
In usually, data that exceeds 3 standard deviations of the mean is considered an outlier

In [ ]:
mean = df['RoomService'].mean()
std = df['RoomService'].std()
outlier = abs(df['RoomService']-mean)>3*std
outliersum = outlier.sum()

<a id="4.1"></a>
### <span style="color:#e76f51;"> Discretization and Binning</span>
group ages into discrete age buckets

In [ ]:
bins = [1,18, 25, 35, 60, 80] # 1-18,19-25,26-35,36-60,61-80
agebin = pd.cut(df['Age'], bins)
# You can override the default interval-based bin labeling by passing a list or array to the labels option:
group_names = ["Kid","Youth", "YoungAdult", "MiddleAged", "Senior"]
agelabel = pd.cut(df['Age'],bins,labels=group_names)

<a id="4.1"></a>
### <span style="color:#e76f51;"> Categorical Data</span>
You can improve `performance` and `memory use` by converting data to Categoricals Types !

In [ ]:
HP_categorical = df['HomePlanet'].astype('category') #from object to category

print(f"Before memory use :{df['HomePlanet'].memory_usage(deep=True)}\n After memory use: {HP_categorical.memory_usage(deep=True)}")
%timeit HP_categorical.value_counts()
%timeit df['HomePlanet'].value_counts()

The comparison shows that performance can be improved by converting data types ！

In [ ]:
HP_categorical.values # Get the value of series of Categorical type 
HP_categorical.values.codes # or HP_categorical.cat.codes
HP_categorical.values.categories # or HP_categorical.cat.categories
map1 = dict(enumerate(HP_categorical.values.categories)) # get the label map

Let's encode `Destination` features
* using Sklearn labelencoder
* using pandas 

In [ ]:
import copy
from sklearn.preprocessing import LabelEncoder
tutorial = df.copy()
tutorial2 = df.copy()

In [ ]:
%time
tutorial['Destination'] = tutorial['Destination'].astype(str)
tutorial['Destination'] = LabelEncoder().fit_transform(tutorial['Destination'])

In [ ]:
%time
HP_categorical = tutorial2['Destination'].astype('category')
tutorial2['Destination'] = HP_categorical.cat.codes

map1 = dict(enumerate(HP_categorical.values.categories))

<a id="4"></a>
# **<center><span style="color:#00BFC4;"> Modeling </span></center>**
Now I take `spaceship-titanic` as an example to show the whole process of data analysis

In [ ]:
# quick overview
ProfileReport(df)

In [ ]:
# Read csv file
train_df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test_df = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

Overview on `train&test` dataset

In [ ]:
print(f"there are total {train_df.shape[0]} samples in train dataset with {train_df.shape[1]-1} features and 1 label to predict")

print(f"\nthere are total {test_df.shape[0]} samples in train dataset with {test_df.shape[1]-1} features")

## <font color='#9966FF'> Missing Values <font><a class='anchor' id='top'></a>

In [ ]:
train_sum = train_df.isna().sum().reset_index().rename(columns={0:'train_count'}).drop(index=13)
test_sum = test_df.isna().sum().reset_index().rename(columns={0:'test_count'}).drop(columns='index')
all_missing = pd.concat([train_sum,test_sum],axis=1)

In [ ]:
fig = px.histogram(
    all_missing,
    x='index',
    y=['train_count','test_count'],
    barmode='group',
    color_discrete_sequence = [mycolors[0],mycolors[4]]
)
fig.update_traces(
    marker_line_width=1
)
fig.update_layout(
    title_text = '<b>Missing values in Train&Test dataset',
    title_font = dict(color = mycolors[2],family="Times New Roman",size=25),
    title_x = 0.5,
    height=500
)
fig.show()
pio.write_image(fig,'Missing values in Train&Test dataset.png',format='png',scale=2.5)

In [ ]:
types = []
for col in train_df.columns:
    dtype = str(train_df[col].dtype)
    types.append(dtype)
datatype_df = pd.DataFrame(types,train_df.columns).rename(columns={0:'type'})
objectl = list(datatype_df.query('type=="object"').index)
floatl = list(datatype_df.query('type=="float64"').index)

For `Numerical Features`, we used sklearn method to fill in missing values

For `Categorical Features`, we used pandas method to fill in missing values

In [ ]:
print(f"Categorical Features: {objectl} \nNumerical Features: {floatl}")

In [ ]:
# Numerical Features
num_impute_col = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
num_imputer = KNNImputer(n_neighbors=5)
train_df[num_impute_col] = num_imputer.fit_transform(train_df[num_impute_col])
test_df[num_impute_col] = num_imputer.fit_transform(test_df[num_impute_col])
#Categorical Features
train_df = train_df.fillna(axis=0,method='ffill')
test_df = test_df.fillna(axis=0,method='ffill')

## <font color='#9966FF'> Age Discretization  <font><a class='anchor' id='top'></a>

In [ ]:
fig = px.histogram(
    train_df,
    x='Age',
    marginal='box',
)
fig.update_traces(
    marker=dict(color=mycolors[0],line_width=1),
)
fig.update_layout(
    title_text = '<b>Age Distribution',
    title_font = dict(color = mycolors[2],family="Times New Roman",size=25),
    title_x = 0.5,
    height=500
)
fig.show()
pio.write_image(fig,'Age Distribution in train set.png',format='png',scale=2.5)

## <font color='#9966FF'> Encoding  <font><a class='anchor' id='top'></a>

In [ ]:
# "Age"
def age_encode(df):
    bins = [1,18,25,35,60,80]
    bin_names = ["Kid","Youth", "YoungAdult", "MiddleAged", "Senior"]
    df['Age'] = pd.cut(df['Age'],bins,labels=bin_names)
    df['Age'] = LabelEncoder().fit_transform(df['Age'])
    return df

age_encode(train_df)
age_encode(test_df)

# "HomePlanet", "CryoSleep","Cabin", "Destination" ,"VIP"
label_cols = ["HomePlanet", "CryoSleep","Cabin", "Destination" ,"VIP"]
def label_encoder(train_df,test_df,columns):
    for col in columns:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)
        train_df[col] = LabelEncoder().fit_transform(train_df[col])
        test_df[col] =  LabelEncoder().fit_transform(test_df[col])
    return train_df, test_df

train_df ,test_df = label_encoder(train_df,test_df ,label_cols)

In [ ]:
train_df = train_df.drop(columns=['PassengerId','Name'])
test_df = test_df.drop(columns=['PassengerId','Name'])

## <font color='#9966FF'> Dataset Pipeline  <font><a class='anchor' id='top'></a>

In [ ]:
# dataset split
train_df,val_df = train_test_split(train_df,test_size=0.2)

In [ ]:
# Create tensorflow datasets
def dataframe_to_dataset(dataframe, shuffle=True, batch_size=32):
  dataframe = dataframe.copy()
  labels = dataframe.pop('Transported')
  ds = tf.data.Dataset.from_tensor_slices((dataframe, labels))
  if shuffle:
    ds = ds.shuffle(buffer_size=len(dataframe))
  ds = ds.batch(batch_size)
  ds = ds.prefetch(batch_size)
  return ds

In [ ]:
train_ds = dataframe_to_dataset(train_df)
val_ds = dataframe_to_dataset(val_df)

## <font color='#9966FF'> Training  <font><a class='anchor' id='top'></a>

In [ ]:
# use simple model
model = models.Sequential([
        layers.Dense(units=16, activation='relu', input_shape=[11,]),
        layers.Dense(units=32, activation='relu'),
        layers.Dense(units=8, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.Dense(units=1, activation='sigmoid')
])
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(),#设置为True时，在模型定义中不需要加sigmoid
              metrics=["accuracy"])
history = model.fit(train_ds, epochs=50, validation_data=val_ds)

In [ ]:
his_df = pd.DataFrame(history.history)
fig = px.line(
    his_df,
    y=['val_accuracy','accuracy'],
    markers=True
)
fig.update_traces(
)
fig.update_layout(
    title_text = '<b> accuracy and loss  of the training  and the validation stage ',
    title_font = dict(color = mycolors[2],family="Times New Roman",size=25),
    title_x = 0.5,
    height=500
)
fig.show()

## <font color='#9966FF'> Predicting <font><a class='anchor' id='top'></a>

In [ ]:
predictions = model.predict(test_df)

In [ ]:
sub = pd.read_csv('../input/spaceship-titanic/sample_submission.csv')
sub['Transported'] = predictions
sub['Transported'] = sub['Transported'].map(lambda x:True if x>=0.5 else False)
sub.to_csv('submission.csv',index=False)

<a id="4"></a>
# **<center><span style="color:#00BFC4;"> Reference </span></center>**

[1.Getting Started with pandas](https://wesmckinney.com/book/pandas-basics.html#pandas_reindex)

[2.Load CSV data](https://www.tensorflow.org/tutorials/load_data/csv)